# Ch 14 · Lab 5 — Checkpoint + EarlyStopping 조합

원본: `12_Wine_Check_and_Stop.py`

두 콜백을 같이 쓰면: 
- **EarlyStopping**: 더 학습할 가치가 없을 때 자동 중단 → 시간 절약
- **ModelCheckpoint**: 그 사이 best 모델을 파일로 저장 → 사용 시 신뢰

이 둘은 역할이 *다르므로* 항상 같이 쓰는 게 표준 패턴.

## 0. 환경

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

from keras.callbacks import ModelCheckpoint, EarlyStopping

## 1. 데이터 (15% 샘플)

In [ ]:
DATA = "../../data/wine.csv"
df_full = pd.read_csv(DATA, header=None)
df = df_full.sample(frac=0.15, random_state=0).reset_index(drop=True)  # 일부러 작게
X = df.iloc[:, 0:12].to_numpy(dtype="float32")
y = df.iloc[:, 12].to_numpy(dtype="float32")
print("샘플링 후 X:", X.shape)

## 2. 두 콜백 모두 등록

In [ ]:
def build_model():
    return Sequential([
        Input(shape=(12,)),
        Dense(30, activation="relu"),
        Dense(12, activation="relu"),
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

MODEL_DIR = "../../outputs/ch14-wine/lab5/"
os.makedirs(MODEL_DIR, exist_ok=True)
ckpt_path = MODEL_DIR + "{epoch:02d}-{val_loss:.4f}.keras"

callbacks = [
    ModelCheckpoint(filepath=ckpt_path, monitor="val_loss", verbose=1, save_best_only=True),
    EarlyStopping(monitor="val_loss", patience=100, verbose=1),
]

keras.utils.set_random_seed(0)
model = build_model()
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

## 3. 학습

In [ ]:
hist = model.fit(X, y, validation_split=0.2, epochs=3500, batch_size=500,
                 verbose=0, callbacks=callbacks)

n_actual = len(hist.history["loss"])
print(f"\n실제 학습 에포크: {n_actual} / 3500  (EarlyStopping 발동)")

## 4. 저장된 체크포인트

In [ ]:
files = sorted(os.listdir(MODEL_DIR))
print(f"saved {len(files)} checkpoints:")
for f in files[-5:]:
    print(f"  {f}")
print(f"... (마지막 5개만 표시)" if len(files) > 5 else "")
print(f"\nbest val_loss: {min(hist.history['val_loss']):.4f}")

## 마무리 — Wine 5 lab 비교
| Lab | 데이터 | 콜백 | 역할 |
| - | - | - | - |
| 1 | 전체 | 없음 | 베이스 |
| 2 | 전체 | Checkpoint | best 저장 |
| 3 | 15% | 없음 | 과적합 *관찰* |
| 4 | 15% | EarlyStopping | 과적합 *방지* |
| 5 | 15% | Checkpoint + EarlyStopping | 표준 조합 |